# Nexus 2.0 Notebook Replica

Este notebook replica una arquitectura tipo Nexus 2.0 usando:
- Foundry / Azure OpenAI compatible endpoint
- Power BI REST API para ejecutar DAX
- Orquestador multiagente en Python

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

repo_root = Path.cwd().parent  # si estás dentro de notebooks/
load_dotenv(repo_root / ".env")

print(os.getenv("AZURE_AI_FOUNDRY_ENDPOINT"))
print(os.getenv("AZURE_AI_FOUNDRY_DEPLOYMENT"))

key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
print(key[:10] + "..." if key else "API KEY NOT FOUND")

https://aoai-pocai-eastus2-dev-01.openai.azure.com/
gpt-5-mini
1IyXRttuSk...


In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # si estás en /nexus2.0/notebooks
sys.path.insert(0, str(repo_root))

print(repo_root)

c:\Users\SamuelAguilarRamirez\nexus2.0


In [3]:
import src.llm_client
print(src.llm_client.__file__)

c:\Users\SamuelAguilarRamirez\nexus2.0\src\llm_client.py


In [4]:
from src.llm_client import AzureAIFoundry

llm = AzureAIFoundry()

response = llm.chat(
    system_prompt="You are a test assistant.",
    user_prompt="Respond OK only."
)

print(response)

OK


In [5]:
from pathlib import Path

path = Path(r"C:\Users\SamuelAguilarRamirez\nexus2.0\src\agents\intent_clarifier.py")

text = path.read_text(encoding="utf-8")
text = text.replace(
    "from prompts.intent_clarifier import INTENT_SYSTEM_PROMPT",
    "from src.prompts.intent_clarifier import INTENT_SYSTEM_PROMPT"
)
path.write_text(text, encoding="utf-8")

print("Fixed import")

Fixed import


In [6]:
import importlib
import src.agents.intent_clarifier
importlib.reload(src.agents.intent_clarifier)

from src.agents.intent_clarifier import IntentClarifierAgent

In [7]:
intent = IntentClarifierAgent(
    llm,
    general_syn="NSR = Net Sales Revenue; market = Ship To",
    dav="Available measures: NSR, Volume. Available dimensions: Period, Ship To, Product, Channel."
).run("Show NSR YTD by channel for Colombia")

print(intent)

{'intent': 'semantic_query', 'agents': [{'name': 'FHB_dataset', 'instruction': 'Retrieve Net Sales Revenue (NSR = SELL-IN) Year-to-date for the current year from the Metrics-Actuals-Rev measures. Filter: Ship To = Colombia. Aggregate NSR YTD by Channel (use Channel dimension). Use Period for the YTD calculation. Return a table with columns: Channel, NSR_YTD (numeric). Scenario: Actuals. Do not apply any additional implicit filters.'}, {'name': 'VisualizationAgent', 'instruction': "Using the returned table (Channel, NSR_YTD), produce a chart that displays NSR_YTD by Channel. Use Channel on the categorical axis and NSR_YTD on the value axis. Sort channels by NSR_YTD descending. Format the value axis as currency and include the title 'NSR YTD by Channel — Colombia'. Choose an appropriate categorical chart (e.g., bar/column) for clear comparison and return JSON plotting instructions."}], 'needs_visualization': True, 'output_format': 'chart', 'business_question': 'Net Sales Revenue (NSR / S

In [8]:
fhb_instruction = intent["agents"][0]["instruction"]
print(fhb_instruction)

Retrieve Net Sales Revenue (NSR = SELL-IN) Year-to-date for the current year from the Metrics-Actuals-Rev measures. Filter: Ship To = Colombia. Aggregate NSR YTD by Channel (use Channel dimension). Use Period for the YTD calculation. Return a table with columns: Channel, NSR_YTD (numeric). Scenario: Actuals. Do not apply any additional implicit filters.


In [ ]:
from src.agents.dax_query_developer import DaxQueryDeveloperAgent

developer = DaxQueryDeveloperAgent(
    llm,
    general_syn="NSR = Net Sales Revenue; market = Ship To",
    dav = """
Tables and columns available:

Period:
- Period[Date]
- Period[Year]
- Period[Month]
- Period[Week]

Ship To:
- Ship To[Country]

Channel:
- Channel[Channel]

Measures:
- [NSR]
- [NSR YTD] (predefined measure)
Available measures: NSR, Volume. Available dimensions: Period, Ship To, Product, Channel."""
)

dax_plan = developer.run(fhb_instruction)

print(dax_plan)

In [ ]:
from src.agents.dax_validator import DaxValidatorAgent

semantic_context = """
Target model: NSR LATAM semantic model

Tables and columns available:

Period:
- Period[Date]
- Period[Year]
- Period[Month]
- Period[Week]

Ship To:
- Ship To[Country]

Channel:
- Channel[Channel]

Measures:
- [NSR]
- [NSR YTD]

Rules:
- NSR is SELL-IN
- Use Period[Date] for time intelligence
"""

validator = DaxValidatorAgent(
    llm_client=llm,
    semantic_context=semantic_context
)

validation_result = validator.run(
    business_question="Get NSR YTD by channel for Colombia",
    dax_query=dax_plan 
)

print(validation_result)

NOT APPROVED

Required fixes:
1. The ORDER BY clause is invalid as written. Replace "ORDER BY [NSR_YTD] DESC" with an ordering that references the result column position or a valid column expression. For this query, use "ORDER BY 2 DESC" to sort by the second column (the ADDCOLUMNS alias "NSR_YTD").
2. Ensure the model contains the referenced measure and columns exactly as named: [NSR YTD], Period[Year], 'Channel'[Channel], and 'Ship To'[Country]. If any name differs in the NSR LATAM model, correct the identifiers accordingly.

Instructions for DAX Developer:
- Fix ONLY the issues listed above.
- Do NOT change any other part of the query.
- Preserve original intent.


In [ ]:
test_query = "EVALUATE VALUES('Reporting View')"

df_test = executor.run(test_query)
display(df_test.head())

In [ ]:
import os
from src.connections.nsr_conn import AdomdConnector
from src.agents.dax_executor import DaxExecutorAgent

BASE_DIR = r"C:\Users\SamuelAguilarRamirez\nexus2.0\src"

PATH_DLL = os.path.join(
    BASE_DIR,
    "lib",
    "Microsoft.AnalysisServices.AdomdClient.dll"
)

STR_CONN = (
    "Provider=MSOLAP;"
    "Data Source=powerbi://api.powerbi.com/v1.0/myorg/NSR LATAM [Test];"
    "Initial Catalog=NSR LATAM Cube;"
    "Integrated Security=ClaimsToken;"
)

nsr_conn = AdomdConnector(PATH_DLL, STR_CONN)

executor = DaxExecutorAgent(nsr_conn)
test_query = "EVALUATE VALUES('Reporting View')"

df_test = executor.run(test_query)
display(df_test.head())

ModuleNotFoundError: No module named 'clr'